In [1]:
import os
import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [4]:
# ---- 1. Load full pairs set ----
pairs = pd.read_csv('../data/processed/abt_buy_pairs.csv')
print(f"Total pairs to classify: {len(pairs)}")

Total pairs to classify: 2194


In [5]:
# ---- 2. Few-shot + SKU-guidance prompt (from Day 7) ----
def classify_pair_v2(name_a, name_b, max_retries=3):
    prompt = f"""You are comparing two product listings to determine if they refer to the same product.

Pay close attention to shared model numbers or SKU codes (alphanumeric codes like '2349B001'). An exact code match is strong evidence of the same product, even if descriptive wording differs significantly.

Example: Product A: "Canon Deluxe Grey Leather Case - 2349B001". Product B: "Canon PSC-1000 Semi-Hard Leather Case - 2349B001". Answer: MATCH (same model code 2349B001).

Example: Product A: "Sony Black Headphones - MDR200". Product B: "Panasonic Blue Speaker - RQ500". Answer: NO_MATCH (different brands, different codes).

Now classify this pair:
Product A: {name_a}
Product B: {name_b}

Respond with only one word: MATCH or NO_MATCH."""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=5
            )
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            return None, str(e)

def parse_response(raw_response):
    if raw_response is None:
        return None
    r = raw_response.upper()
    if r == "MATCH":
        return 1
    elif r == "NO_MATCH":
        return 0
    elif "NO_MATCH" in r:
        return 0
    elif "MATCH" in r:
        return 1
    return None

In [7]:
# ---- 3. Run with incremental saving ----
output_path = '../data/processed/abt_buy_llm_fewshot_preds.csv'
results = []
errors = []

for idx, row in tqdm(pairs.iterrows(), total=len(pairs)):
    raw_response, error = classify_pair_v2(row['name_abt'], row['name_buy'])
    pred_label = parse_response(raw_response)

    if error:
        errors.append({'idx': idx, 'id_abt': row['id_abt'], 'id_buy': row['id_buy'], 'error': error})

    results.append({
        'id_abt': row['id_abt'],
        'id_buy': row['id_buy'],
        'name_abt': row['name_abt'],
        'name_buy': row['name_buy'],
        'label': row['label'],
        'raw_response': raw_response,
        'pred_label': pred_label
    })

    if (idx + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(output_path, index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(output_path, index=False)
print(f"\nSaved {len(results_df)} predictions to {output_path}")

100%|██████████| 2194/2194 [29:01<00:00,  1.26it/s]


Saved 2194 predictions to ../data/processed/abt_buy_llm_fewshot_preds.csv


In [8]:
# ---- 4. Report errors/parsing failures ----
n_errors = len(errors)
n_unparsed = results_df['pred_label'].isna().sum()
print(f"API errors: {n_errors}")
print(f"Unparsed responses: {n_unparsed}")
if errors:
    print("\nSample errors:")
    for e in errors[:5]:
        print(e)

API errors: 0
Unparsed responses: 0


In [9]:
# ---- 5. Metrics ----
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

valid = results_df.dropna(subset=['pred_label'])
print(f"\nScoring on {len(valid)} / {len(results_df)} pairs")

p = precision_score(valid['label'], valid['pred_label'])
r = recall_score(valid['label'], valid['pred_label'])
f1 = f1_score(valid['label'], valid['pred_label'])
acc = accuracy_score(valid['label'], valid['pred_label'])

print(f"\nLLM Few-Shot + SKU Guidance Results:")
print(f"Precision: {p:.3f}")
print(f"Recall:    {r:.3f}")
print(f"F1:        {f1:.3f}")
print(f"Accuracy:  {acc:.3f}")


Scoring on 2194 / 2194 pairs

LLM Few-Shot + SKU Guidance Results:
Precision: 0.994
Recall:    0.846
F1:        0.914
Accuracy:  0.920


In [10]:
# ---- 6. Full comparison table ----
comparison = pd.DataFrame([
    {'Method': 'Rule-based (Jaccard, t=0.2)', 'Precision': 0.994, 'Recall': 0.910, 'F1': 0.950, 'Accuracy': 0.952},
    {'Method': 'LLM zero-shot', 'Precision': 0.999, 'Recall': 0.902, 'F1': 0.948, 'Accuracy': 0.951},
    {'Method': 'LLM few-shot + SKU guidance', 'Precision': round(p, 3), 'Recall': round(r, 3), 'F1': round(f1, 3), 'Accuracy': round(acc, 3)},
])
print("\n--- Full Comparison Table ---")
print(comparison.to_string(index=False))
comparison.to_csv('../data/processed/abt_buy_comparison_table.csv', index=False)


--- Full Comparison Table ---
                     Method  Precision  Recall    F1  Accuracy
Rule-based (Jaccard, t=0.2)      0.994   0.910 0.950     0.952
              LLM zero-shot      0.999   0.902 0.948     0.951
LLM few-shot + SKU guidance      0.994   0.846 0.914     0.920
